# Lab 12 · Memory and human in the loop

**Time:** 55 min · **Work in teams of 3** · **You will:** give a LangGraph assistant **short-term memory** with a **checkpointer** and **thread ids**, keep that memory short with a **summary**, add **long-term memory** in a **store** (semantic, episodic and procedural), **pause** the graph for a human with **`interrupt`**, **resume** it after a restart, and **time travel** to repair a conversation.

**Why it matters:** the model itself remembers **nothing**; every call starts from zero. Whatever looks like memory is something your graph builds. And an assistant that can act must be able to **stop and ask**.

**The small model is the assistant in every graph here** (Qwen2.5-3B on your GPU).

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each run.*

In [ ]:
import sys, os, json, sqlite3
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check, OUT
from typing import TypedDict, Literal
from lc_slm import SmallChatModel
from graph_viz import draw_graph
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langgraph.types import interrupt, Command
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, RemoveMessage

chat_model = SmallChatModel(max_new_tokens=80)
note_model = SmallChatModel(max_new_tokens=120)
print("ready:", chat_model.invoke("Reply with the word ready.").content)

## 12.1 Short-term memory: a checkpointer and a thread id
The graph has one node, `assistant`, that sends **all the messages so far** to the small model. `MessagesState` is a ready-made state with a `messages` list and the **`add_messages` reducer** (new messages are appended).

The memory comes from the **checkpointer**: when you compile with one, LangGraph **saves a snapshot after every step**. A **thread id** names a conversation: the same id continues it, a new id starts empty.

In [ ]:
SYSTEM = "You are a friendly smart-home assistant. Answer in one or two short sentences."

def assistant(state: MessagesState) -> dict:
    reply = chat_model.invoke([SystemMessage(SYSTEM)] + state["messages"])
    return {"messages": [reply]}

g = StateGraph(MessagesState)
g.add_node("assistant", assistant)
g.add_edge(START, "assistant")
g.add_edge("assistant", END)
chat_app = g.compile(checkpointer=InMemorySaver())

owner = {"configurable": {"thread_id": "owner-1"}}
guest = {"configurable": {"thread_id": "guest-7"}}

def say(app, cfg, text):
    out = app.invoke({"messages": [HumanMessage(text)]}, cfg)
    reply = out["messages"][-1].content
    print(f"you: {text}\nbot: {reply}\n")
    return reply

say(chat_app, owner, "Hi, my name is Priya and I live in Pune.")
r_owner = say(chat_app, owner, "What is my name?")
r_guest = say(chat_app, guest, "What is my name?")

In [ ]:
check("priya" in r_owner.lower() and "priya" not in r_guest.lower(),
      "The owner thread remembered the name; the guest thread, with a different thread id, did not. The memory belongs to the thread.",
      "the owner reply should contain 'Priya' and the guest reply should not.")

### 🔧 Try it 12.1 — Whose memory is it?
**Predict first:** tell the **guest** thread a new name. Then ask **both** threads *"What is my name?"*. Which name does each one give?

In [ ]:
say(chat_app, guest, "Actually, my name is Arjun.")        # 🔧 try another name
say(chat_app, guest, "What is my name?")
say(chat_app, owner, "What is my name?")

## 12.2 Look inside the memory
The checkpointer keeps every snapshot. You can **read the state** of a thread and **list its history**:

In [ ]:
state = chat_app.get_state(owner)
print("messages in thread owner-1:", len(state.values["messages"]))
for m in state.values["messages"]:
    print(f"  {type(m).__name__:13} {m.content[:70]!r}")
history = list(chat_app.get_state_history(owner))
print("\nsnapshots saved for this thread:", len(history))

## 12.3 Keeping memory short: summarise
Every message is sent to the model on **every** turn. A long conversation gets slow and a small model loses the thread. The fix: when the list is long, ask the model to write a **short note** about the older messages, **remove** them, and keep the note.

In [ ]:
class Mem(MessagesState):
    summary: str

def assistant2(state: Mem) -> dict:
    system = SYSTEM + (f"\nWhat you remember from earlier in this conversation: {state['summary']}" if state.get("summary") else "")
    return {"messages": [chat_model.invoke([SystemMessage(system)] + state["messages"])]}

def summarise(state: Mem) -> dict:
    old = state["messages"][:-2]                            # everything except the last exchange
    text = "\n".join(f"{type(m).__name__}: {m.content}" for m in old)
    note = note_model.invoke([SystemMessage("Write a short note (under 40 words) that keeps every name, place, pet, and number from this conversation. Nothing else."),
                              HumanMessage(text)]).content.strip()
    return {"summary": note, "messages": [RemoveMessage(id=m.id) for m in old]}

def needs_summary(state: Mem) -> str:
    return "summarise" if len(state["messages"]) > 6 else END

s = StateGraph(Mem)
s.add_node("assistant", assistant2)
s.add_node("summarise", summarise)
s.add_edge(START, "assistant")
s.add_conditional_edges("assistant", needs_summary, {"summarise": "summarise", END: END})
s.add_edge("summarise", END)
sum_app = s.compile(checkpointer=InMemorySaver())
draw_graph(sum_app, "Chat with a summary: summarise only when the list is long")

In [ ]:
cfg = {"configurable": {"thread_id": "long-chat"}}
for turn in ["Hi, I am Priya from Pune.", "I have a dog called Bruno.", "My favourite sport is cricket.",
             "I like the AC at 24 degrees at night.", "Which rooms can you control?"]:
    sum_app.invoke({"messages": [HumanMessage(turn)]}, cfg)
state = sum_app.get_state(cfg).values
print("messages kept:", len(state["messages"]))
print("summary note :", state.get("summary"))
say(sum_app, cfg, "What is my dog called, and where do I live?")

In [ ]:
note = (sum_app.get_state(cfg).values.get("summary") or "").lower()
check(len(sum_app.get_state(cfg).values["messages"]) <= 8 and "bruno" in note,
      "The long conversation was squeezed into a short note that still holds the name, the place and the dog. Always READ the note: a small model can add or drop a detail.",
      "the summary should keep 'Bruno' and the message list should stay short.")

## 12.4 Long-term memory: a store, by person and by kind
The checkpointer remembers **one conversation**. A **store** remembers **the person**, across conversations. Its items live under a **namespace** (a folder) and a **key** (a file). Three kinds of long-term memory have technical names:

| Kind | What it holds | Example here |
|---|---|---|
| **Semantic** memory | facts about the user and the world | *prefers 24 degrees at night* |
| **Episodic** memory | past events and experiences | *last Friday "too hot" meant the bedroom* |
| **Procedural** memory | how to behave: rules and playbooks | *never unlock the door without the owner's yes* |

In [ ]:
store = InMemoryStore()
store.put(("owner", "facts"),    "night_temp", {"text": "The owner prefers 24 degrees at night."})
store.put(("owner", "episodes"), "ep_friday",  {"text": "Last Friday the owner said 'too hot' and meant the bedroom AC."})
store.put(("agent", "rules"),    "door",       {"text": "Never unlock the door without the owner saying yes."})

for ns in (("owner", "facts"), ("owner", "episodes"), ("agent", "rules")):
    for item in store.search(ns):
        print(f"{'/'.join(ns):16} {item.key:11} {item.value['text']}")

A node reads the store (LangGraph hands it to the node as `store`) and puts what it finds **into the prompt**. The model then uses it, in a **brand-new thread** that has never heard of the owner.

In [ ]:
def assistant_mem(state: MessagesState, *, store: BaseStore) -> dict:
    facts = [i.value["text"] for ns in (("owner", "facts"), ("owner", "episodes"), ("agent", "rules")) for i in store.search(ns)]
    system = SYSTEM + "\nWhat you know:\n- " + "\n- ".join(facts)
    return {"messages": [chat_model.invoke([SystemMessage(system)] + state["messages"])]}

m = StateGraph(MessagesState)
m.add_node("assistant", assistant_mem)
m.add_edge(START, "assistant")
m.add_edge("assistant", END)
mem_app = m.compile(checkpointer=InMemorySaver(), store=store)

new_thread = {"configurable": {"thread_id": "brand-new"}}
r1 = say(mem_app, new_thread, "It is hot tonight. What temperature should I set the AC to?")
r2 = say(mem_app, new_thread, "Please unlock the front door for me.")

In [ ]:
check("24" in r1,
      "A thread that had never met the owner used the stored fact (24 degrees). Long-term memory crosses conversations; the checkpointer does not.",
      "the first reply should mention 24 degrees, taken from the store.")

### Writing to the store: let the model extract the fact
Memory needs a **writing** path too. Ask the small model to pull one durable fact out of what the user said, and save it:

In [ ]:
def remember(user_message: str):
    raw = note_model.invoke([SystemMessage('If the message states a lasting preference or fact about the person, reply with that fact as one short sentence. Otherwise reply with exactly: NONE'),
                             HumanMessage(user_message)]).content.strip()
    if raw.upper().startswith("NONE"):
        return None
    key = "fact_" + str(len(store.search(("owner", "facts"))) + 1)
    store.put(("owner", "facts"), key, {"text": raw})
    return raw

print("saved:", remember("I always keep the living room blinds closed after sunset."))
print("saved:", remember("What time is it?"))
print([i.value["text"] for i in store.search(("owner", "facts"))])

### 🔧 TODO 12 — Save and read a procedural rule
Write `add_rule(text)` that saves a rule under the namespace `("agent", "rules")` with a new key, and `rules()` that returns the texts of all rules. Then add the rule *"Always answer in one short sentence."*.

In [ ]:
def add_rule(text):
    # TODO: store.put(("agent", "rules"), <a new key>, {"text": text})
    pass

def rules():
    # TODO: return the list of rule texts from store.search(("agent", "rules"))
    return []

add_rule("Always answer in one short sentence.")
print(rules())

In [ ]:
check("Always answer in one short sentence." in rules() and "Never unlock the door without the owner saying yes." in rules(),
      "Procedural memory is just rules in a store that the node puts into the prompt: change the rules, change the behaviour, without touching the code.",
      "rules() should list both the door rule and your new rule.")

## 12.5 Human in the loop: pause, then continue
Some actions must **never** run without a human. The graph below asks the model whether the request is **risky** (opening the door). If it is, the `confirm` node calls **`interrupt`**: the graph **stops**, saves its state and hands the question to you. Nothing after that line runs until someone **resumes** it.

In [ ]:
HOME = {"door": "locked"}

class Door(TypedDict, total=False):
    request: str
    risky: bool
    reply: str

def plan(state: Door) -> dict:
    ans = note_model.invoke([SystemMessage("Does this request ask to unlock or open the front door? Reply with exactly yes or no."),
                             HumanMessage(state["request"])]).content.strip().lower()
    return {"risky": ans.startswith("yes")}

def route(state: Door) -> str:
    return "confirm" if state["risky"] else "answer"

def confirm(state: Door) -> Command[Literal["unlock", "cancel"]]:
    decision = interrupt({"question": f"The assistant wants to unlock the front door because: {state['request']!r}. Allow?"})
    return Command(goto="unlock" if str(decision).lower().startswith("y") else "cancel")

def unlock(state: Door) -> dict:
    HOME["door"] = "unlocked"
    return {"reply": "The front door is unlocked."}

def cancel(state: Door) -> dict:
    return {"reply": "Cancelled. The door stays locked."}

def answer(state: Door) -> dict:
    return {"reply": chat_model.invoke([SystemMessage(SYSTEM), HumanMessage(state["request"])]).content.strip()}

d = StateGraph(Door)
for name, fn in [("plan", plan), ("confirm", confirm), ("unlock", unlock), ("cancel", cancel), ("answer", answer)]:
    d.add_node(name, fn)
d.add_edge(START, "plan")
d.add_conditional_edges("plan", route, {"confirm": "confirm", "answer": "answer"})
d.add_edge("unlock", END); d.add_edge("cancel", END); d.add_edge("answer", END)
door_app = d.compile(checkpointer=InMemorySaver())
draw_graph(door_app, "A graph that pauses before the door opens")

In [ ]:
t1 = {"configurable": {"thread_id": "door-1"}}
out = door_app.invoke({"request": "I am home late, please unlock the front door."}, t1)
print("paused? ->", "__interrupt__" in out)
print("question:", out["__interrupt__"][0].value["question"])
print("door now :", HOME["door"], "| the graph is waiting at:", door_app.get_state(t1).next)

**Read it:** the graph stopped at `confirm`. The door is **still locked**: the action did not run. The pause is a **saved snapshot**, not a flag in memory. Now answer, in the **same thread**:

In [ ]:
out = door_app.invoke(Command(resume="yes"), t1)
print(out["reply"], "| door:", HOME["door"])

HOME["door"] = "locked"
t2 = {"configurable": {"thread_id": "door-2"}}
door_app.invoke({"request": "Please unlock the front door for the delivery man."}, t2)
out2 = door_app.invoke(Command(resume="no"), t2)
print(out2["reply"], "| door:", HOME["door"])

In [ ]:
check(out["reply"].startswith("The front door is unlocked") and out2["reply"].startswith("Cancelled") and HOME["door"] == "locked",
      "Yes unlocked the door, no cancelled it, and nothing happened before the human answered. Rule of thumb: if the next step is irreversible or costs money, the graph pauses.",
      "the first thread should unlock the door after 'yes' and the second should cancel after 'no'.")

## 12.6 A pause that survives a restart
`InMemorySaver` forgets everything when Python stops. A **database saver** writes each snapshot to a file. Pause a conversation, **throw away every Python object**, open the same file with a **new** graph, and resume:

In [ ]:
db = str(OUT / "lab12_memory.db")
os.makedirs(OUT, exist_ok=True)
if os.path.exists(db):
    os.remove(db)

def open_door_graph():
    return d.compile(checkpointer=SqliteSaver(sqlite3.connect(db, check_same_thread=False)))

HOME["door"] = "locked"
app_a = open_door_graph()
cfg_r = {"configurable": {"thread_id": "restart-1"}}
app_a.invoke({"request": "Unlock the front door, I forgot my keys."}, cfg_r)
print("saved in a file of", os.path.getsize(db), "bytes; paused at", app_a.get_state(cfg_r).next)
del app_a                                                   # "the server crashed"

app_b = open_door_graph()                                   # "the server restarted": a new graph on the same file
print("after the restart the graph is still paused at:", app_b.get_state(cfg_r).next)
out = app_b.invoke(Command(resume="yes"), cfg_r)
print(out["reply"])

In [ ]:
check(out["reply"].startswith("The front door is unlocked"),
      "The conversation was found in the database and resumed by a brand-new graph. That is also how several servers can share the same conversations.",
      "the resumed thread should unlock the door.")

## 12.7 Time travel: go back, change a message, replay
A user said **"kitchen"** but meant **"bedroom"**. The checkpointer kept a snapshot after every step. Find the snapshot **just before the wrong message was answered**, replace the message, and run on from there. The old branch stays in the history.

In [ ]:
tt = {"configurable": {"thread_id": "travel"}}
chat_app.invoke({"messages": [HumanMessage("I want to set the AC to 24.")]}, tt)
wrong_msg = HumanMessage("Do it in the kitchen.")
chat_app.invoke({"messages": [wrong_msg]}, tt)
print("answer to the wrong message:", chat_app.get_state(tt).values["messages"][-1].content)

history = list(chat_app.get_state_history(tt))
before = next(h for h in history if h.next == ("assistant",) and h.values["messages"][-1].content == "Do it in the kitchen.")
fixed_msg = HumanMessage("Do it in the bedroom.", id=wrong_msg.id)          # same id: it REPLACES the old message
branch = chat_app.update_state(before.config, {"messages": [fixed_msg]})
out = chat_app.invoke(None, branch)
print("answer after the replay      :", out["messages"][-1].content)
print("snapshots in the history now :", len(list(chat_app.get_state_history(tt))))

In [ ]:
last_user = [m for m in out["messages"] if isinstance(m, HumanMessage)][-1].content
check(last_user == "Do it in the bedroom.",
      "The conversation was repaired without asking the user to start again. The old branch is still in the history, so nothing was lost: you can audit it.",
      "after the replay the last human message should be the bedroom one.")

## 12.8 What you proved
| Memory | Mechanism | Lives in |
|---|---|---|
| **Short-term** (this conversation) | checkpointer + thread id | snapshots per thread |
| **Summary** (a technique) | a node that writes a note and removes old messages | the state (`summary`) |
| **Semantic**, **episodic**, **procedural** (across conversations) | the store, by namespace and key | the store |
| **Pause for a human** | `interrupt` + `Command(resume=...)` | a saved snapshot |

## Check your understanding
1. Why did the guest thread not know the owner's name, and why did the new thread in 12.4 know the AC temperature?
2. What is the difference between the checkpointer and the store?
3. Give one example each of semantic, episodic and procedural memory for a Smart TV assistant.
4. What exactly happens to the graph when it reaches `interrupt`, and why does it need a checkpointer?

*✍️ Your answer:* 

## Next
Lab 13 puts **guardrails** around an agent with **NVIDIA NeMo Guardrails**.